In [2]:
from google.colab import files
uploaded = files.upload()

Saving gateway.csv to gateway.csv
Saving ledger.csv to ledger.csv


In [9]:
import pandas as pd

# Load the datasets
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

print("Files loaded successfully!")

Files loaded successfully!


In [12]:
# Check for nulls in Ledger
print("--- Nulls in Ledger ---")
print(ledger.isnull().sum())

# Check for nulls in Gateway
print("\n--- Nulls in Gateway ---")
print(gateway.isnull().sum())

--- Nulls in Ledger ---
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

--- Nulls in Gateway ---
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [13]:
# Check for duplicate IDs in Ledger
ledger_dupes = ledger[ledger.duplicated('transaction_id', keep=False)]
print(f"Duplicate IDs in Ledger: {len(ledger_dupes)}")
if len(ledger_dupes) > 0:
    display(ledger_dupes.sort_values(by='transaction_id'))

# Check for duplicate IDs in Gateway
gateway_dupes = gateway[gateway.duplicated('transaction_id', keep=False)]
print(f"\nDuplicate IDs in Gateway: {len(gateway_dupes)}")
if len(gateway_dupes) > 0:
    display(gateway_dupes.sort_values(by='transaction_id'))

Duplicate IDs in Ledger: 0

Duplicate IDs in Gateway: 0


In [14]:
# Identify IDs in Ledger that are NOT in Gateway
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]

print(f"Found {len(missing_in_gateway)} records missing in Gateway:")
display(missing_in_gateway)

Found 2 records missing in Gateway:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


In [15]:
# Identify IDs in Gateway that are NOT in Ledger
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])]

print(f"Found {len(missing_in_ledger)} records missing in Ledger:")
display(missing_in_ledger)

Found 1 records missing in Ledger:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,2026-03-05,M003,1800.0,success,Card


In [16]:
# Merge only records that exist in both files
merged = pd.merge(ledger, gateway, on="transaction_id", how="inner", suffixes=("_ledger", "_gateway"))

# Filter where the 'amount_usd' columns don't match
amount_mismatches = merged[merged['amount_usd_ledger'] != merged['amount_usd_gateway']]

# Select only relevant columns for the report
report_cols = ['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']
print(f"Found {len(amount_mismatches)} amount mismatches:")
display(amount_mismatches[report_cols])

Found 2 amount mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway
1,R002,850.0,900.0
6,R008,640.0,600.0


In [17]:
# Re-using the 'merged' dataframe from the previous step
status_mismatches = merged[merged['status_ledger'] != merged['status_gateway']]

# Select only relevant columns for the report
report_cols = ['transaction_id', 'status_ledger', 'status_gateway']
print(f"Found {len(status_mismatches)} status mismatches:")
display(status_mismatches[report_cols])

Found 1 status mismatches:


,transaction_id,status_ledger,status_gateway
3,R005,success,failed


In [18]:
# Perform the full reconciliation merge
recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

# Add a 'variance' column
recon['amount_variance'] = recon['amount_usd_ledger'] - recon['amount_usd_gateway']

# Add a flag for mismatches
recon['is_discrepancy'] = (
    (recon['_merge'] != 'both') |
    (recon['amount_usd_ledger'] != recon['amount_usd_gateway']) |
    (recon['status_ledger'] != recon['status_gateway'])
)

# Filter for the final exception report
exception_report = recon[recon['is_discrepancy'] == True]

print("--- FINAL EXCEPTION REPORT ---")
display(exception_report[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'status_ledger', 'status_gateway', '_merge']])
exception_report.to_csv('final_reconciliation_report.csv', index=False)

--- FINAL EXCEPTION REPORT ---


,transaction_id,amount_usd_ledger,amount_usd_gateway,status_ledger,status_gateway,_merge
1,R002,850.0,900.0,success,success,both
3,R004,2100.0,NaN,success,NaN,left_only
4,R005,7200.0,7200.0,success,failed,both
7,R008,640.0,600.0,success,success,both
9,R010,2500.0,NaN,success,NaN,left_only
10,R011,NaN,1800.0,NaN,success,right_only


In [20]:
# Calculations
metrics = {
    "Total Ledger Records": len(ledger),
    "Total Gateway Records": len(gateway),
    "Missing in Gateway (Count)": len(recon[recon['_merge'] == 'left_only']),
    "Missing in Gateway (Value)": recon[recon['_merge'] == 'left_only']['amount_usd_ledger'].sum(),
    "Missing in Ledger (Count)": len(recon[recon['_merge'] == 'right_only']),
    "Missing in Ledger (Value)": recon[recon['_merge'] == 'right_only']['amount_usd_gateway'].sum(),
    "Amount Mismatches": len(recon[(recon['_merge'] == 'both') & (recon['amount_usd_ledger'] != recon['amount_usd_gateway'])]),
    "Status Mismatches": len(recon[(recon['_merge'] == 'both') & (recon['status_ledger'] != recon['status_gateway'])])
}

# Convert to a clean display table
metrics_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
print("\n--- RECONCILIATION SUMMARY METRICS ---")
display(metrics_df)


--- RECONCILIATION SUMMARY METRICS ---


,Metric,Value
0,Total Ledger Records,10.0
1,Total Gateway Records,9.0
2,Missing in Gateway (Count),2.0
3,Missing in Gateway (Value),4600.0
4,Missing in Ledger (Count),1.0
5,Missing in Ledger (Value),1800.0
6,Amount Mismatches,2.0
7,Status Mismatches,1.0


In [44]:
# 1. Define the filename
json_file_name = 'summary_metrics.json'

# 2. Write the dictionary to a file
# 'indent=4' makes the JSON file "pretty" and easy to read
with open(json_file_name, 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"File '{json_file_name}' has been created.")

File 'summary_metrics.json' has been created.


In [45]:

files.download(json_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Export results to CSV for download
missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('amount_mismatches.csv', index=False)
status_mismatches.to_csv('status_mismatches.csv', index=False)

In [24]:
from google.colab import files

files.download("missing_in_gateway.csv")
files.download("missing_in_ledger.csv")
files.download("amount_mismatches.csv")
files.download("status_mismatches.csv")
files.download("final_reconciliation_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import json
from google.colab import files

# Upload the file
print("Please upload api_response_sample.json:")
uploaded = files.upload()


Please upload api_response_sample.json:


Saving api_response_sample.json to api_response_sample.json


In [26]:
# Load the JSON data
with open('api_response_sample.json', 'r') as f:
    data = json.load(f)

print("JSON file loaded successfully!")

JSON file loaded successfully!


In [28]:
# Flatten the data
df_flattened = pd.json_normalize(data)

# Preview the flattened structure
print("Flattened Data Preview:")
display(df_flattened.head())

Flattened Data Preview:


,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [30]:
# Create a deep copy to protect the original 'df_flattened'
df_clean = df_flattened.copy()

print("Original data backed up into 'df_clean'. You can now modify it safely.")

Original data backed up into 'df_clean'. You can now modify it safely.


In [37]:
# 1. Convert to lowercase
# 2. Replace dots (common in JSON flattening) with underscores
# 3. Replace spaces with underscores
df_clean.columns = [
    col.strip().lower().replace('.', '_').replace(' ', '_')
    for col in df_clean.columns
]

print("Cleaned Column Names:")
print(df_clean.columns.tolist())

Cleaned Column Names:
['generated_at', 'source', 'batches']


In [38]:
# Keywords to look for in column names
date_keywords = ['date', 'time', 'at', 'timestamp']

# Loop through columns and convert if a keyword is found
for col in df_clean.columns:
    if any(key in col for key in date_keywords):
        print(f"Converting column: {col}")
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# Verify the conversion
print("\nNew Data Types:")
print(df_clean.dtypes)

Converting column: generated_at
Converting column: batches

New Data Types:
generated_at    datetime64[ns, UTC]
source                       object
batches              datetime64[ns]
dtype: object


In [39]:
# Save the cleaned, normalized copy to a CSV
output_file_name = 'normalized_api_data.csv'

df_clean.to_csv(output_file_name, index=False)

print(f"File '{output_file_name}' has been created in Colab storage.")

File 'normalized_api_data.csv' has been created in Colab storage.


In [40]:
from google.colab import files

files.download(output_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>